In [1]:
pip install gudhi

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import gudhi as gd
import matplotlib.pyplot as plt
import networkx as nx
import ipywidgets as widgets
from IPython.display import display

In [ ]:
# Generate a symmetric distance matrix

def load_distance_matrix(file_name):
    return pd.read_excel(file_name, index_col=0).values

# Function to compute Rips complex from a distance matrix
def compute_rips_from_distance_matrix(distance_matrix, max_edge_length):
    rips_complex = gd.RipsComplex(distance_matrix=distance_matrix, max_edge_length=max_edge_length)
    simplex_tree = rips_complex.create_simplex_tree(max_dimension=2)
    return simplex_tree

# Function to extract simplices
def extract_simplices(simplex_tree):
    simplices = {'vertices': set(), 'edges': [], 'faces': []}

    for simplex, _ in simplex_tree.get_filtration():
        if len(simplex) == 1:
            simplices['vertices'].add(simplex[0])
        elif len(simplex) == 2:
            simplices['edges'].append(tuple(simplex))
        elif len(simplex) == 3:
            simplices['faces'].append(tuple(simplex))

    return simplices

# Function to compute persistence using the Elder Rule
def compute_persistence(simplex_tree):
    parent = {}  # Union-Find structure to track components
    birth_time = {}  # Track birth times of components
    birth_death_dict = {}

    for simplex, filtration in simplex_tree.get_filtration():
        if len(simplex) == 1:  
            v = simplex[0]
            parent[v] = v  # Each vertex is its own component initially
            birth_time[v] = 0  # Birth time is 0
            birth_death_dict[v] = (0, None)  # Birth at filtration value 0, death unknown

    def find(v):
        if parent[v] != v:
            parent[v] = find(parent[v])  # Path compression
        return parent[v]
    
    def union(v1, v2, death_time):
        root1 = find(v1)
        root2 = find(v2)

        if root1 != root2:  # Only merge if different components
            if birth_time[root1] < birth_time[root2]:  
                parent[root2] = root1
                birth_death_dict[root2] = (birth_death_dict[root2][0], death_time)
            else:
                parent[root1] = root2
                birth_death_dict[root1] = (birth_death_dict[root1][0], death_time)

    for simplex, filtration in simplex_tree.get_filtration():
        if len(simplex) == 2:  
            v1, v2 = simplex
            union(v1, v2, filtration)

    return birth_death_dict

# Function to visualize the Rips complex and display persistence intervals
def visualize_rips_complex_and_persistence(radius):
    simplex_tree = compute_rips_from_distance_matrix(distance_matrix, radius)
    simplices = extract_simplices(simplex_tree)
    birth_death_dict = compute_persistence(simplex_tree)

    G = nx.Graph()
    G.add_nodes_from(simplices['vertices'])
    G.add_edges_from(simplices['edges'])

    pos = nx.spring_layout(G, seed=42)
    labels = {i: f"P {i+1}" for i in simplices['vertices']}

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].set_title(f"Vietoris-Rips Complex (Max Edge = {radius:.2f})")
    nx.draw_networkx_nodes(G, pos, ax=axes[0], node_size=10, node_color="red")
    nx.draw_networkx_edges(G, pos, ax=axes[0], edgelist=simplices['edges'], width=2, edge_color="blue")
    
    for face in simplices['faces']:
        pts = np.array([pos[face[0]], pos[face[1]], pos[face[2]]])
        axes[0].fill(pts[:, 0], pts[:, 1], color='cyan', alpha=0.3)

    #nx.draw_networkx_labels(G, pos, labels, font_size=5, font_color="black", ax=axes[0])
    axes[0].axis("off")

    axes[1].set_title("Persistence Intervals ($H_0$)")
    y_positions = list(range(1, n + 1))  # Dynamically adjust y-axis range
    
    for i, (birth, death) in enumerate(birth_death_dict.values()):
        y = y_positions[i]  # Keep original protein order
        axes[1].plot([birth, death if death else 1.2], [y, y], 'b-', lw=2)
        axes[1].scatter(birth, y, color='red', zorder=3)
        if death:
            axes[1].scatter(death, y, color='black', zorder=3)
    
    axes[1].set_xlabel("Filtration Value")
    axes[1].set_ylabel("Protein")
    axes[1].set_yticks(range(1, n + 1))  # Dynamic y-axis ticks
    axes[1].grid(True)

    plt.show()

    print("\nPersistence Intervals:")
    for i, (birth, death) in birth_death_dict.items():
        print(f"Protein {i+1}: Birth = {birth}, Death = {death if death else 'infinity'}")

if __name__ == "__main__":
    distance_matrix_file = "rbf_distance_matrix_trial.xlsx"
    distance_matrix = load_distance_matrix(distance_matrix_file)
    n = distance_matrix.shape[0]
    radius_slider = widgets.FloatSlider(value=0.2, min=0.0, max=np.max(distance_matrix), step=0.01, description='Max Edge Length:')
    interactive_plot = widgets.interactive(visualize_rips_complex_and_persistence, radius=radius_slider)

    display(interactive_plot)

interactive(children=(FloatSlider(value=0.2, description='Max Edge Length:', max=1.578902147363058, step=0.01)…